In [ ]:
import os

# A mappa neve
folder = "datafiles"

# Fájlok és mappák listázása
files = os.listdir(folder)

print(files)


In [ ]:
import pandas as pd

for file in files:
    name = file.split('.')[0]
    print(f"\n{name}:")

    globals()[f"{name}"] = pd.read_csv(f"datafiles/{file}")
    print(globals()[f"{name}"].sample(5))

In [ ]:
import pandas as pd

# Csak a befejezett versenyek (pozíció nem null)
results_finished = results[results['positionText'] != 'R'].copy()
results_finished['position'] = pd.to_numeric(results_finished['position'], errors='coerce')

# Qualifying és results összekapcsolása
df = qualifying.merge(results_finished, on=['raceId', 'driverId', 'constructorId'], how='inner')

# Verseny infók hozzáadása
df = df.merge(races[['raceId', 'year', 'name', 'circuitId']], on='raceId', how='left')

# Pálya infók hozzáadása
df = df.merge(circuits[['circuitId', 'name', 'country']], on='circuitId', how='left', suffixes=('_race', '_circuit'))

# Versenyző infók
df = df.merge(drivers[['driverId', 'forename', 'surname', 'nationality']], on='driverId', how='left')
df['driver_name'] = df['forename'] + ' ' + df['surname']
df['driver_nationality'] = df['nationality']

# Konstruktőr infók
df = df.merge(constructors[['constructorId', 'name', 'nationality']], on='constructorId', how='left', suffixes=('', '_constructor'))
df['constructor_name'] = df['name']
df['constructor_nationality'] = df['nationality_constructor']

# Pozíció változás számítás
df['position_change'] = df['position_x'] - df['position_y']  # position_x=quali, position_y=race

# Évtized kategória
df['decade'] = (df['year'] // 10) * 10

# Grid pozíció kategória
df['grid_category'] = pd.cut(df['grid'], bins=[0, 3, 10, 30], labels=['Top3', 'Midfield', 'Back'])

# Kontinens (egyszerűsített)
continent_map = {
    'UK': 'Europe', 'Italy': 'Europe', 'Germany': 'Europe', 'France': 'Europe', 
    'Spain': 'Europe', 'Monaco': 'Europe', 'Belgium': 'Europe', 'Austria': 'Europe',
    'Netherlands': 'Europe', 'Portugal': 'Europe', 'Hungary': 'Europe', 'Russia': 'Europe',
    'USA': 'Americas', 'Brazil': 'Americas', 'Canada': 'Americas', 'Mexico': 'Americas',
    'Argentina': 'Americas',
    'Japan': 'Asia', 'China': 'Asia', 'Malaysia': 'Asia', 'Singapore': 'Asia',
    'Korea': 'Asia', 'Bahrain': 'Asia', 'UAE': 'Asia', 'Saudi Arabia': 'Asia',
    'Australia': 'Oceania',
    'South Africa': 'Africa'
}
df['continent'] = df['country'].map(continent_map).fillna('Other')

# Végleges adattábla
final_df = df[[
    'raceId', 'name_race', 'year', 'decade', 'name_circuit', 'country', 'continent',
    'driverId', 'driver_name', 'driver_nationality',
    'constructorId', 'constructor_name', 'constructor_nationality',
    'position_x', 'position_y', 'grid', 'grid_category',
    'position_change', 'points', 'fastestLap', 'rank'
]].rename(columns={
    'name_race': 'race_name',
    'name_circuit': 'circuit_name',
    'position_x': 'quali_position',
    'position_y': 'race_position'
})

# Mentés
final_df.to_csv('f1_quali_vs_race.csv', index=False)

print(f"Kész! {len(final_df)} sor adattal.")
print("\nElső 5 sor:")
display(final_df.sample(5))
print("\nOszlopok:")
display(final_df.info())